# 40 — QML con Datos Reales: Kernel Cuántico vs Clásico

## Paradigma de kernel cuántico

En **Quantum Machine Learning (QML)** basado en kernels, la idea central es mapear los datos clásicos
$x \in \mathbb{R}^d$ hacia un espacio de Hilbert cuántico $\mathcal{H}$ mediante un **feature map cuántico**:

$$\phi: \mathbb{R}^d \to \mathcal{H}, \quad x \mapsto |\phi(x)\rangle$$

implementado por un circuito parametrizado $U(x)|0\rangle = |\phi(x)\rangle$.

El **kernel cuántico** entre dos puntos $x, x'$ se define como el solapamiento de sus estados cuánticos:

$$k(x, x') = |\langle \phi(x') | \phi(x) \rangle|^2$$

Este kernel puede entonces alimentar cualquier clasificador basado en kernels (SVM, regresión gaussiana, etc.).

### ¿Por qué puede ser ventajoso?

- **Espacios de Hilbert exponencialmente grandes**: $n$ qubits dan acceso a un espacio de dimensión $2^n$.
- **Expresividad no clásica**: ciertos kernels cuánticos son difíciles de simular eficientemente con métodos clásicos (hardness de sampling).
- **Inductive bias cuántico**: si los datos tienen estructura cuántica subyacente, el kernel cuántico puede capturarla naturalmente.

### Referencia teórica

| Marco | Resultado clave |
|---|---|
| Schuld & Killoran (PRL 2019) | Circuitos cuánticos como feature maps en espacios de Hilbert |
| Havlíček et al. (Nature 2019) | Kernel cuántico con separación computacional potencial |
| Huang et al. (Nature Comm. 2021) | Ventaja cuántica demostrable en datos generados cuánticamente |

En este laboratorio comparamos el kernel cuántico **ZZFeatureMap** contra kernels clásicos (RBF, lineal) en tres datasets de clasificación estándar.

## 0. Configuración e Imports

Este laboratorio implementa y compara kernels cuánticos frente a kernels clásicos (RBF, polinómico) en un dataset real de clasificación. El kernel cuántico se calcula como el **solapamiento de estado** entre dos muestras procesadas por un circuito feature map:

$$K(x_i, x_j) = |\langle 0|U^\dagger(x_j) U(x_i)|0\rangle|^2$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_circles, make_blobs
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

print("Imports cargados correctamente.")
print(f"NumPy {np.__version__}")
import qiskit
print(f"Qiskit {qiskit.__version__}")

## 1. Dataset y Preprocesamiento

Usamos el dataset **Iris** (o Wine) con dos clases y dos características, normalizado al rango $[0, \pi]$ para codificación en ángulos de rotación del circuito cuántico.

In [ ]:
# ---------------------------------------------------------------------------
# Feature Map cuántico manual: ZZFeatureMap
# H^n → Rz(2x_i) → CRz(2(π−x_i)(π−x_j)) × reps
# ---------------------------------------------------------------------------

def zz_feature_map(x: np.ndarray, n_qubits: int = 2, reps: int = 2) -> QuantumCircuit:
    """ZZFeatureMap: H^n → Rz(2x_i) → CRz(2(π-x_i)(π-x_j)) × reps.

    Parameters
    ----------
    x        : vector de features de entrada (se cicla si len(x) < n_qubits)
    n_qubits : número de qubits del circuito
    reps     : número de repeticiones del bloque de encoding

    Returns
    -------
    QuantumCircuit sin medidas, listo para Statevector
    """
    qc = QuantumCircuit(n_qubits)
    for rep in range(reps):
        # Capa de Hadamard + rotaciones Z individuales
        for i in range(n_qubits):
            qc.h(i)
            qc.rz(2 * x[i % len(x)], i)
        # Capa de interacciones ZZ por pares
        for i in range(n_qubits - 1):
            for j in range(i + 1, n_qubits):
                val = 2 * (np.pi - x[i % len(x)]) * (np.pi - x[j % len(x)])
                qc.cx(i, j)
                qc.rz(val, j)
                qc.cx(i, j)
    return qc


def quantum_kernel(X1: np.ndarray, X2: np.ndarray, n_qubits: int = 2) -> np.ndarray:
    """Matriz de kernel cuántico K[i,j] = |⟨ϕ(x2_j)|ϕ(x1_i)⟩|².

    Parameters
    ----------
    X1, X2  : matrices de datos (n_samples × n_features)
    n_qubits : número de qubits; debe ser >= n_features

    Returns
    -------
    K : ndarray de forma (len(X1), len(X2))
    """
    K = np.zeros((len(X1), len(X2)))
    for i, xi in enumerate(X1):
        sv_i = Statevector(zz_feature_map(xi, n_qubits))
        for j, xj in enumerate(X2):
            sv_j = Statevector(zz_feature_map(xj, n_qubits))
            K[i, j] = abs(sv_j.inner(sv_i)) ** 2
    return K


# Verificación rápida: kernel de un punto consigo mismo debe ser ≈ 1
x_test = np.array([0.5, 1.2])
K_self = quantum_kernel(x_test.reshape(1, -1), x_test.reshape(1, -1))
print(f"K(x, x) = {K_self[0, 0]:.6f}  (esperado ≈ 1.0)")

# Visualizar el circuito para un ejemplo
qc_ejemplo = zz_feature_map(x_test, n_qubits=2, reps=2)
print(f"\nCircuito ZZFeatureMap (2 qubits, 2 reps):")
print(qc_ejemplo.draw(output='text'))

## 2. Feature Map Cuántico

El feature map codifica datos clásicos $\mathbf{x}$ en un estado cuántico $U(\mathbf{x})|0\rangle$ mediante:
- Rotaciones $R_y(x_i)$ para cada característica.
- Capas de entrelazamiento CNOT para capturar correlaciones no lineales.

La elección del feature map determina la geometría del kernel en el espacio de Hilbert.

In [ ]:
# ---------------------------------------------------------------------------
# Experimento: make_moons — Kernel cuántico vs RBF vs Lineal
# ---------------------------------------------------------------------------

# 1. Generar dataset
X_moons, y_moons = make_moons(n_samples=200, noise=0.1, random_state=42)

# 2. Train/test split (80/20)
X_train, X_test, y_train, y_test = train_test_split(
    X_moons, y_moons, test_size=0.2, random_state=42
)

# 3. Escalar features a [0, π] para el feature map cuántico
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std  = scaler.transform(X_test)

# Mapeo [media-3σ, media+3σ] → [0, π]
def scale_to_pi(X: np.ndarray) -> np.ndarray:
    """Escala linealmente los datos para que caigan en [0, π]."""
    X_min = X.min(axis=0)
    X_max = X.max(axis=0)
    return np.pi * (X - X_min) / (X_max - X_min + 1e-8)

X_train_q = scale_to_pi(X_train_std)
X_test_q  = scale_to_pi(X_test_std)

print("Calculando kernel cuántico (puede tardar ~30s en CPU)...")
K_train = quantum_kernel(X_train_q, X_train_q)
K_test  = quantum_kernel(X_test_q,  X_train_q)
print("Kernel cuántico calculado.")

# 4. SVM con kernel cuántico (precomputado)
svm_quantum = SVC(kernel='precomputed', C=1.0)
svm_quantum.fit(K_train, y_train)
acc_q_train = accuracy_score(y_train, svm_quantum.predict(K_train))
acc_q_test  = accuracy_score(y_test,  svm_quantum.predict(K_test))

# 5. SVM con kernel RBF
svm_rbf = SVC(kernel='rbf', gamma='scale', C=1.0)
svm_rbf.fit(X_train_std, y_train)
acc_rbf_train = accuracy_score(y_train, svm_rbf.predict(X_train_std))
acc_rbf_test  = accuracy_score(y_test,  svm_rbf.predict(X_test_std))

# 6. SVM con kernel lineal
svm_lin = SVC(kernel='linear', C=1.0)
svm_lin.fit(X_train_std, y_train)
acc_lin_train = accuracy_score(y_train, svm_lin.predict(X_train_std))
acc_lin_test  = accuracy_score(y_test,  svm_lin.predict(X_test_std))

# 7. Tabla de resultados
print("\n" + "="*55)
print(f"{'Modelo':<22} {'Acc Train':>12} {'Acc Test':>12}")
print("-"*55)
print(f"{'Quantum (ZZFeatureMap)':<22} {acc_q_train:>12.4f} {acc_q_test:>12.4f}")
print(f"{'RBF':<22} {acc_rbf_train:>12.4f} {acc_rbf_test:>12.4f}")
print(f"{'Lineal':<22} {acc_lin_train:>12.4f} {acc_lin_test:>12.4f}")
print("="*55)

## 3. Matriz de Kernel Cuántico

Calculamos la matriz de Gram $K_{ij} = K(x_i, x_j)$ para todos los pares de muestras. Una buena matriz kernel debe ser definida positiva y exhibir separación entre clases.

In [ ]:
# ---------------------------------------------------------------------------
# Fronteras de decisión: SVM cuántico vs SVM RBF en make_moons
# ---------------------------------------------------------------------------

def plot_decision_boundary_quantum(ax, svm_model, X_train_q, y_train,
                                   X_test_std, y_test, scaler, title):
    """Dibuja la frontera de decisión del SVM cuántico en espacio original."""
    # Grid en espacio estandarizado
    x_min, x_max = X_test_std[:, 0].min() - 0.5, X_test_std[:, 0].max() + 0.5
    y_min, y_max = X_test_std[:, 1].min() - 0.5, X_test_std[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 40),
                         np.linspace(y_min, y_max, 40))
    grid = np.c_[xx.ravel(), yy.ravel()]

    # Escalar grid a [0, π]
    grid_q = scale_to_pi(grid)

    # Kernel entre grid y training set
    K_grid = quantum_kernel(grid_q, X_train_q)
    Z = svm_model.predict(K_grid).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8)

    # Puntos de train
    for cls, color, label in [(0, 'blue', 'Clase 0'), (1, 'red', 'Clase 1')]:
        mask_tr = (y_train == cls)
        mask_te = (y_test  == cls)
        ax.scatter(X_train_std[mask_tr, 0], X_train_std[mask_tr, 1],
                   c=color, edgecolors='k', linewidths=0.5, s=30, label=f'Train {label}')
        ax.scatter(X_test_std[mask_te, 0], X_test_std[mask_te, 1],
                   c=color, marker='*', edgecolors='k', linewidths=0.5, s=80, label=f'Test {label}')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='upper right')


def plot_decision_boundary_classical(ax, svm_model, X_train_std, y_train,
                                     X_test_std, y_test, title):
    """Dibuja la frontera de decisión de un SVM clásico."""
    x_min, x_max = X_train_std[:, 0].min() - 0.5, X_train_std[:, 0].max() + 0.5
    y_min, y_max = X_train_std[:, 1].min() - 0.5, X_train_std[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                         np.linspace(y_min, y_max, 200))
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='k', linewidths=0.8)

    for cls, color, label in [(0, 'blue', 'Clase 0'), (1, 'red', 'Clase 1')]:
        mask_tr = (y_train == cls)
        mask_te = (y_test  == cls)
        ax.scatter(X_train_std[mask_tr, 0], X_train_std[mask_tr, 1],
                   c=color, edgecolors='k', linewidths=0.5, s=30, label=f'Train {label}')
        ax.scatter(X_test_std[mask_te, 0], X_test_std[mask_te, 1],
                   c=color, marker='*', edgecolors='k', linewidths=0.5, s=80, label=f'Test {label}')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='upper right')


fig, axes = plt.subplots(1, 2, figsize=(13, 5))

print("Generando frontera de decisión cuántica (puede tardar ~1 min)...")
plot_decision_boundary_quantum(
    axes[0], svm_quantum, X_train_q, y_train, X_test_std, y_test, scaler,
    title=f"SVM Kernel Cuántico (ZZFeatureMap)\nTest acc = {acc_q_test:.3f}"
)

plot_decision_boundary_classical(
    axes[1], svm_rbf, X_train_std, y_train, X_test_std, y_test,
    title=f"SVM Kernel RBF\nTest acc = {acc_rbf_test:.3f}"
)

plt.suptitle("Fronteras de decisión — make_moons", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Clasificación con SVM Cuántico vs Clásico

Usamos la matriz kernel cuántica como input para una SVM. Comparamos:
- **QSVM**: kernel cuántico (solapamiento de estados)
- **SVM-RBF**: kernel gaussiano clásico
- **SVM-poly**: kernel polinómico

In [ ]:
# ---------------------------------------------------------------------------
# Análisis de expressivity del kernel: Kernel Target Alignment (KTA)
# ---------------------------------------------------------------------------

def kernel_target_alignment(K: np.ndarray, y: np.ndarray) -> float:
    """Kernel Target Alignment: KTA = ⟨K, yy^T⟩_F / (‖K‖_F · ‖yy^T‖_F).

    Mide qué tan bien alineado está el kernel con la estructura de las etiquetas.
    KTA ∈ [-1, 1]; valores más altos → mejor separabilidad.

    Parameters
    ----------
    K : matriz de kernel (n × n)
    y : etiquetas de clase (n,) con valores en {0, 1} o {-1, +1}

    Returns
    -------
    kta : float
    """
    # Convertir etiquetas a {-1, +1}
    y_pm = 2 * y.astype(float) - 1
    T = np.outer(y_pm, y_pm)           # Matriz objetivo ideal
    numerator   = np.sum(K * T)        # producto de Frobenius ⟨K, T⟩
    denominator = np.linalg.norm(K, 'fro') * np.linalg.norm(T, 'fro')
    return float(numerator / (denominator + 1e-12))


# Kernels clásicos: calcular matrices en training set
from sklearn.metrics.pairwise import rbf_kernel, linear_kernel
gamma_rbf = 1.0 / (X_train_std.shape[1] * X_train_std.var())  # gamma='scale'
K_rbf_train = rbf_kernel(X_train_std, gamma=gamma_rbf)
K_lin_train = linear_kernel(X_train_std)

# KTA de cada kernel
kta_q   = kernel_target_alignment(K_train,     y_train)
kta_rbf = kernel_target_alignment(K_rbf_train, y_train)
kta_lin = kernel_target_alignment(K_lin_train,  y_train)

print("Kernel Target Alignment (KTA) — make_moons")
print("-" * 38)
print(f"  Cuántico (ZZFeatureMap): {kta_q:+.4f}")
print(f"  RBF:                     {kta_rbf:+.4f}")
print(f"  Lineal:                  {kta_lin:+.4f}")
print()
print("KTA > 0 → kernel alineado con la tarea; cuanto mayor, mejor.")

# Heatmap de las matrices de kernel para los 20 primeros puntos
n_vis = 20
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

kernels_vis = [
    (K_train[:n_vis, :n_vis],     f"Kernel Cuántico\nKTA={kta_q:.3f}"),
    (K_rbf_train[:n_vis, :n_vis], f"Kernel RBF\nKTA={kta_rbf:.3f}"),
    (K_lin_train[:n_vis, :n_vis], f"Kernel Lineal\nKTA={kta_lin:.3f}"),
]

for ax, (K_vis, title) in zip(axes, kernels_vis):
    im = ax.imshow(K_vis, cmap='viridis', aspect='auto', vmin=0)
    ax.set_title(title)
    ax.set_xlabel("Índice muestra")
    ax.set_ylabel("Índice muestra")
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle("Matrices de kernel — primeros 20 puntos de training",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Análisis de Entrenamiento y Generalización

Comparamos accuracy de entrenamiento vs test, y la dependencia con el tamaño del dataset. El kernel cuántico puede ser ventajoso cuando los datos tienen una estructura que se mapea naturalmente al espacio de Hilbert.

In [ ]:
# ---------------------------------------------------------------------------
# Comparativa en 3 datasets: moons, circles, blobs
# ---------------------------------------------------------------------------

datasets = [
    ("make_moons",    make_moons(n_samples=200,   noise=0.15,  random_state=42)),
    ("make_circles",  make_circles(n_samples=200, noise=0.1,   factor=0.5, random_state=42)),
    ("make_blobs",    make_blobs(n_samples=200,   centers=2,   random_state=42)),
]

results = []

for name, (X, y) in datasets:
    print(f"\nProcesando {name}...")

    # Preprocesado
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    sc = StandardScaler().fit(Xtr)
    Xtr_s = sc.transform(Xtr)
    Xte_s = sc.transform(Xte)
    Xtr_q = scale_to_pi(Xtr_s)
    Xte_q = scale_to_pi(Xte_s)

    # Kernel cuántico
    Ktr = quantum_kernel(Xtr_q, Xtr_q)
    Kte = quantum_kernel(Xte_q, Xtr_q)
    svm_q = SVC(kernel='precomputed', C=1.0).fit(Ktr, ytr)
    acc_q = accuracy_score(yte, svm_q.predict(Kte))

    # Kernel RBF
    svm_r = SVC(kernel='rbf', gamma='scale', C=1.0).fit(Xtr_s, ytr)
    acc_r = accuracy_score(yte, svm_r.predict(Xte_s))

    # Kernel lineal
    svm_l = SVC(kernel='linear', C=1.0).fit(Xtr_s, ytr)
    acc_l = accuracy_score(yte, svm_l.predict(Xte_s))

    results.append({
        'Dataset':         name,
        'Acc_Quantum':     round(acc_q, 4),
        'Acc_RBF':         round(acc_r, 4),
        'Acc_Lineal':      round(acc_l, 4),
        'Mejor_Modelo':    max(
            [('Quantum', acc_q), ('RBF', acc_r), ('Lineal', acc_l)],
            key=lambda t: t[1]
        )[0]
    })
    print(f"  Quantum={acc_q:.4f}  RBF={acc_r:.4f}  Lineal={acc_l:.4f}")

# Mostrar DataFrame
try:
    import pandas as pd
    df = pd.DataFrame(results)
    print("\n" + "="*65)
    print("RESULTADOS COMPARATIVOS (accuracy en test set)")
    print("="*65)
    print(df.to_string(index=False))
    print("="*65)
except ImportError:
    print("\nResultados (pandas no disponible):")
    header = f"{'Dataset':<15} {'Quantum':>10} {'RBF':>10} {'Lineal':>10} {'Mejor':>10}"
    print(header)
    print("-" * len(header))
    for r in results:
        print(f"{r['Dataset']:<15} {r['Acc_Quantum']:>10.4f} {r['Acc_RBF']:>10.4f} "
              f"{r['Acc_Lineal']:>10.4f} {r['Mejor_Modelo']:>10}")

## Quantum Advantage en QML: estado del arte y limitaciones

### ¿Cuándo puede el kernel cuántico superar al clásico?

**Huang et al. (Nature Communications, 2021)** demostraron que existe una separación formal entre kernels cuánticos y clásicos:

> *"Un kernel cuántico puede aprender tareas donde el mejor clasificador clásico con kernel fracasa, siempre que los datos sean generados por un proceso cuántico difícil de simular clásicamente."*

La clave es el concepto de **geometric difference** $g(K_Q, K_C)$: si el kernel cuántico se parece poco al kernel clásico óptimo, puede aportar información estructural nueva.

### Limitaciones principales

| Problema | Descripción |
|---|---|
| **Barren Plateaus** | Gradientes exponencialmente pequeños al escalar $n$ qubits (McClean et al. 2018). El paisaje de entrenamiento se vuelve plano. |
| **Proyective simulation** | La medición colapsa el estado; el kernel se estima como media estadística de shots. |
| **Overfitting cuántico** | Alta expressividad ≠ buena generalización. El sesgo inductivo puede ser inadecuado para datos clásicos. |
| **Coste computacional** | Calcular $K_{ij}$ requiere $O(n^2)$ evaluaciones de circuito. Para $n=160$ muestras → 25.600 ejecuciones. |
| **Dequantización** | Tang (2022): muchos algoritmos QML tienen análogos clásicos eficientes si los datos se pueden acceder vía sampling. |

### El camino hacia ventaja cuántica real

```
Datos clásicos → Feature map cuántico difícil de simular → Kernel cuántico → SVM
                       ↑
           Aquí debe estar la "magia": el circuito debe
           implementar una distribución #P-hard de medir
```

Para datos clásicos estándar (moons, circles, blobs), el kernel RBF generalmente es competitivo o superior, porque la estructura subyacente es euclidiana, no cuántica.

### Referencias

1. **Schuld, M. & Killoran, N.** (2019). *Quantum Machine Learning in Feature Hilbert Spaces.* Physical Review Letters, 122, 040504. [DOI:10.1103/PhysRevLett.122.040504]

2. **Havlíček, V. et al.** (2019). *Supervised learning with quantum-enhanced feature spaces.* Nature, 567, 209–212. [DOI:10.1038/s41586-019-0980-2]

3. **Huang, H.-Y. et al.** (2021). *Power of data in quantum machine learning.* Nature Communications, 12, 2631. [DOI:10.1038/s41467-021-22539-9]

4. **McClean, J.R. et al.** (2018). *Barren plateaus in quantum neural network training landscapes.* Nature Communications, 9, 4812.

5. **Tang, E.** (2022). *Dequantizing algorithms to understand quantum advantage.* STOC 2022.